# 07. Jenkins-коннектор и E2E smoke test

Добавляем Jenkins job connector. Генерирует `agents/step_07_jenkins.py`
с полным набором connectors (demo, telegram, jenkins, vk).

Этот ноутбук также включает E2E smoke test: проверяет, что агент видит
Jenkins tools, читает live metadata и (опционально) запускает сборку.

```bash
uv run langgraph dev --config langgraph.steps.json
```

## Визуальная рамка

![Jenkins smoke test flow](visuals/07_jenkins_smoke.svg)

Этот шаг показывает внешний DevOps connector как управляемый smoke: preview, read-only metadata и только затем gated real build.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'pyproject.toml').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

from connectors import CONNECTOR_TOOLS
from connectors.jenkins import trigger_jenkins_job, get_jenkins_job_info

print("Jenkins tools available:", "get_jenkins_job_info" in [t.name for t in CONNECTOR_TOOLS])
print("trigger_jenkins_job:", "trigger_jenkins_job" in [t.name for t in CONNECTOR_TOOLS])

### Dry-run preview

Проверяем, что Jenkins tool формирует правильный payload.

In [ ]:
preview = json.loads(trigger_jenkins_job.invoke({
    'parameters': {'OPENCLAW_SMOKE': 'true'},
    'dry_run': True,
}))
print(json.dumps(preview, indent=2, ensure_ascii=False))
assert preview.get("dry_run"), "Expected dry_run mode"

### Live metadata

Читаем live информацию о Jenkins job (реальный HTTP-запрос).

In [ ]:
info = json.loads(get_jenkins_job_info.invoke({'dry_run': False}))
print(json.dumps(info, indent=2, ensure_ascii=False))
if info.get("ok"):
    print(f"\nJob: {info.get('response', {}).get('name', 'unknown')}")
    print(f"URL: {info.get('job_url', 'unknown')}")

### Real build (gated)

Срабатывает только при `OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1` в `.env`.
После запуска bridge не ждёт завершения сборки — агент сам решит, нужно ли.

In [ ]:
RUN_REAL = os.getenv("OPENCLAW_RUN_REAL_JENKINS_PIPELINE", "0")

if RUN_REAL == "1":
    result = json.loads(trigger_jenkins_job.invoke({
        'parameters': {'OPENCLAW_SMOKE': 'true'},
        'dry_run': False,
    }))
    print(json.dumps(result, indent=2, ensure_ascii=False))
    assert result.get("ok"), "Build trigger failed"
else:
    print("Skipping real build. Set OPENCLAW_RUN_REAL_JENKINS_PIPELINE=1 to enable.")

### Сгенерировать entrypoint

In [ ]:
AGENTS_DIR = REPO_ROOT / "agents"
AGENTS_DIR.mkdir(exist_ok=True)

agent_code = '''"""Step 07: all connectors including Jenkins."""

from __future__ import annotations

import os
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, LocalShellBackend
from dotenv import load_dotenv

from connectors import CONNECTOR_TOOLS

load_dotenv()

DEFAULT_MODEL = "openrouter:tencent/hy3:free"


def _workspace_root() -> Path:
    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()


def _backend():
    root_dir = _workspace_root()
    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":
        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)
    return LocalShellBackend(
        root_dir=root_dir, virtual_mode=True,
        inherit_env=True, timeout=120, max_output_bytes=80_000,
    )


SUBAGENTS = [
    {"name": "repo-researcher", "description": "Map repository structure, APIs, tests, and likely change locations.", "system_prompt": "You research codebases. Inspect files, identify relevant modules, and return concise findings with file paths and rationale."},
    {"name": "reviewer", "description": "Review proposed patches for bugs, missing tests, and regressions.", "system_prompt": "You are a code reviewer. Focus on correctness, edge cases, tests, security, and maintainability. Be specific and concise."},
]

BASE_PROMPT = """\
You are OpenClaw, an experimental open-source coding agent built with LangChain
Deep Agents. You help with software engineering, repository navigation, product
research, implementation, and DevOps checks.

Work like a careful senior engineer:
- inspect before editing;
- keep changes focused;
- verify with tests or equivalent checks;
- explain only the useful result to the user.
"""

SWE_PROMPT = BASE_PROMPT + """\

You are running in SWE mode. Optimize for issue resolution:
- reproduce or characterize the failure first;
- localize the root cause before patching;
- add regression coverage where practical;
- run the narrowest useful verification before broad checks;
- keep a clear chain from issue to patch to test.
"""


agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=CONNECTOR_TOOLS,
    system_prompt=BASE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution", "/skills/product-research"],
    memory=["/AGENTS.md"],
    backend=_backend(),
    interrupt_on={"execute": True},
)

swe_agent = create_deep_agent(
    model=os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL),
    tools=CONNECTOR_TOOLS,
    system_prompt=SWE_PROMPT,
    subagents=SUBAGENTS,
    skills=["/skills/swe-resolution"],
    memory=["/AGENTS.md"],
    backend=_backend(),
    interrupt_on={"execute": True, "write_file": True, "edit_file": True},
)
'''

step_file = AGENTS_DIR / "step_07_jenkins.py"
step_file.write_text(agent_code)

steps_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw": "./agents/step_07_jenkins.py:agent",
        "openclaw_swe": "./agents/step_07_jenkins.py:swe_agent",
    },
    "env": ".env",
}
config_path = REPO_ROOT / "langgraph.steps.json"
config_path.write_text(json.dumps(steps_config, indent=2) + "\n")

print(f"Wrote {step_file}")
print(f"Wrote {config_path}")
print()
print("Restart:")
print("  uv run langgraph dev --config langgraph.steps.json")

### Проверочный prompt

После перезапуска попробуй:

> "Use the Jenkins connector. Check the configured job info, then trigger a real smoke build for the marat job."